# Preprocessing experiments

Tried using the dataset protocol to choose different playing styles, then listened to the actual recordings. The main finding was that the protocol does not consistently describe individual files. These cells keep the investigation, but the inferred technique/fingering columns should not be treated as verified labels.

### Working out the numbering

First tried turning recording numbers into protocol steps and fingering groups. For example, 14 gives step 5 in group 2. This follows the written protocol, not a check of what is actually played.

The remainder (`% 9`) gives the step within a nine-recording group. Whole-number division (`// 9`) gives the fingering group. Subtracting one first makes recording numbers start at the right position.

In [1]:
from pathlib import Path
import sys

# Allow imports when running this notebook from notebooks/ or the project root.
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

recording_number = 14

protocol_step = (recording_number - 1) % 9 + 1
fingering = (recording_number - 1) // 9 + 1

# should return 5 2
print(protocol_step, fingering)

5 2


### Reusing the calculation

Put the calculation in a function so it can be called for each file. The returned technique is only the **intended** technique. Listening later showed that these labels are not reliable enough for evaluating performance by technique.

In [2]:
def get_protocol(recording_number):
    protocol_step = (recording_number - 1) % 9 + 1
    fingering = (recording_number - 1) // 9 + 1

    if protocol_step <= 4:
        technique = "single_strum"
    elif protocol_step <= 6:
        technique = "slow_picking"
    else:
        technique = "repeated strumming"

    return protocol_step, fingering, technique

### Finding the files

Some names contain `f a_series` instead of `fa_series`, and the Bm files start with `Hm`. Removed spaces only in the matching string and used the folder for the chord label. The real paths stay unchanged. This found 600 Fender recordings across 24 chords; noise and the extra GarageBand recordings are outside this table. Matching the expected counts does not verify the playing styles.

The loop below reads the chord folders, fixes spaces only in a temporary filename string, and collects matching Fender files into a table. The final cross-tab counts recordings for each chord and estimated technique.

In [3]:
from pathlib import Path
import pandas as pd

path = ROOT / "data/raw/data/Train"
rows = []

for directory in path.glob("*/"):
    if directory.name == "Noise":
        continue

    for file in directory.glob("*.wav"):
        normalised_name = file.name.replace(" ", "")
        parts = normalised_name.split("_", 1)

        if len(parts) < 2:
            print("Unexpected filename:", file.name)
            continue

        recording_description = parts[1]

        if recording_description.startswith("acoustic_guitar_fender_fa_series_"):
            recording_number = int(file.stem.split("_")[-1])
            protocol_step, fingering, technique = get_protocol(recording_number)

            rows.append({
                "filepath": file,
                "chord": directory.name,
                "recording_number": recording_number,
                "protocol_step": protocol_step,
                "fingering": fingering,
                "technique": technique
            })

metadata = pd.DataFrame(rows)
display(metadata)

print(metadata.shape)
display(pd.crosstab(metadata["chord"], metadata["technique"]))

,filepath,chord,recording_number,protocol_step,fingering,technique
0,C:\Users\Marcus\Desktop\Coding\university\dat1...,A,1,1,1,single_strum
1,C:\Users\Marcus\Desktop\Coding\university\dat1...,A,10,1,2,single_strum
2,C:\Users\Marcus\Desktop\Coding\university\dat1...,A,11,2,2,single_strum
3,C:\Users\Marcus\Desktop\Coding\university\dat1...,A,12,3,2,single_strum
4,C:\Users\Marcus\Desktop\Coding\university\dat1...,A,13,4,2,single_strum
...,...,...,...,...,...,...
595,C:\Users\Marcus\Desktop\Coding\university\dat1...,Gm,5,5,1,slow_picking
596,C:\Users\Marcus\Desktop\Coding\university\dat1...,Gm,6,6,1,slow_picking
597,C:\Users\Marcus\Desktop\Coding\university\dat1...,Gm,7,7,1,repeated strumming
598,C:\Users\Marcus\Desktop\Coding\university\dat1...,Gm,8,8,1,repeated strumming


(600, 6)


technique,repeated strumming,single_strum,slow_picking
chord,,,
A,7,12,6
A#,7,12,6
A#m,7,12,6
Am,7,12,6
B,7,12,6
Bm,7,12,6
C,7,12,6
C#,7,12,6
C#m,7,12,6


### Listening to the original and crop

Selected recordings 1, 5 and 7 of A and Am. Both players use mono at 22,050 Hz so the comparison is fair. Earlier, the native-rate playback sounded brighter; that came from resampling, rather than cropping.

The crop below keeps **up to three seconds**, matching the training pipeline. Shorter clips use the available audio without padding (minimum one second). `info` shows its position in the original file. Invalid or extremely short clips are reported rather than stopping the whole cell.

For each selected file, play the full recording and then the crop at the same sample rate. Print the crop times to help check which part of the recording was kept.

In [4]:
import librosa
from IPython.display import Audio, display
from src.features import DEFAULT_DURATION, preprocess_audio

examples = metadata[
    metadata["chord"].isin(["A", "Am"])
    & metadata["recording_number"].isin([1, 5, 7])
]

for example in examples.itertuples():
    print(
        f"{example.chord} — recording {example.recording_number}"
        f" — protocol estimate: {example.technique}"
    )

    full_audio, sr = librosa.load(example.filepath, sr=22050, mono=True)
    try:
        cropped_audio, sr, info = preprocess_audio(example.filepath, duration=DEFAULT_DURATION)
    except ValueError as exc:
        print(f"Skipped crop: {exc}")
        display(Audio(full_audio, rate=sr, normalize=False))
        continue

    print("Full recording")
    display(Audio(full_audio, rate=sr, normalize=False))

    print(f"Crop: {info["selected_duration_s"]:.2f} seconds (maximum {DEFAULT_DURATION:g})")
    display(Audio(cropped_audio, rate=sr, normalize=False))

    print(info)

A — recording 1 — protocol estimate: single_strum
Full recording


Crop: 3.00 seconds (maximum 3)


{'original_duration_s': 5.6072562358276645, 'start_s': 1.0448979591836736, 'stop_s': 4.044897959183674, 'trim_end_s': 5.352199546485261, 'top_db': 30.0, 'selected_duration_s': 3.0}
A — recording 5 — protocol estimate: slow_picking
Full recording


Crop: 3.00 seconds (maximum 3)


{'original_duration_s': 5.547392290249433, 'start_s': 0.3599092970521542, 'stop_s': 3.359909297052154, 'trim_end_s': 5.547392290249433, 'top_db': 30.0, 'selected_duration_s': 3.0}
A — recording 7 — protocol estimate: repeated strumming
Full recording


Crop: 3.00 seconds (maximum 3)


... (3 outputs left out)

Crop: 3.00 seconds (maximum 3)


{'original_duration_s': 5.7070294784580495, 'start_s': 1.4396371882086167, 'stop_s': 4.439637188208617, 'trim_end_s': 5.7070294784580495, 'top_db': 30.0, 'selected_duration_s': 3.0}
Am — recording 7 — protocol estimate: repeated strumming
Full recording


Crop: 3.00 seconds (maximum 3)


{'original_duration_s': 5.98639455782313, 'start_s': 0.6385487528344671, 'stop_s': 3.6385487528344673, 'trim_end_s': 5.491519274376417, 'top_db': 30.0, 'selected_duration_s': 3.0}


### What came out of this

- Listening found repeated strums in A recording 3 and a single downstrum in A recording 5, despite what the protocol suggested. Stopped trying to classify playing styles from the numbers.
- In A recording 7, the two-second crop cut off a note. Changed the standard crop to **three seconds** to leave more room for the playing and decay. This is a practical choice from listening, not proof that three seconds is best for every recording.
- Still using 22,050 Hz. The change in brightness was audible, but its effect on classification has not been tested.
- A longer rolling window will make future live feedback slower to respond. Shorter file recordings are retained: mean chroma still produces 12 features regardless of frame count.
- Next: try the model with a quiet recording of my own guitar. Speech/singing mixtures and augmentation stay on the backlog.

Corrected training run: all 630 recordings retained (504 train, 126 validation), 95.2% accuracy and 0.951 macro F1. The first implementation incorrectly required three full seconds and rejected 11 clips. It now uses up to three seconds with no padding. Original validation assignments are unchanged; the previously excluded E recording was added to training.